# 04_03 Shortcut connections

*📖 Book: section 4.4*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → [model reads] → model learns → model writes → MCP tool
> ```
>
> You are here: **model reads**
>
> Project files this connects to:
>
> - `src/homer_ai/gpt.py`
>
> **This notebook:** In a tall stack the learning signal fades before it reaches the lower floors. A shortcut that adds the input back keeps it alive.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why a tall stack of layers is hard to train: the learning signal fades on its way down
- What a shortcut is: add the input back to the output
- A real experiment: the size of the gradient in each layer, with and without shortcuts
- That our small `model.py` has no shortcuts, and doesn't need them

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **shortcut connection** | a path that skips a layer: the layer's input is added back to its output (also called a residual connection) | `x = x + layer(x)` |
| **vanishing gradient** | the learning signal getting smaller and smaller on its way back through many layers, until the lowest layers barely learn | a gradient of 0.0002 in layer 1 |

## 3. The idea

### ❓ What problem do shortcuts solve?

Learning ([01_03](../01_foundations/01_03_how_a_program_learns.ipynb)) works backwards. The mistake at the output makes a **gradient** (a signal that says how to change each weight), and that signal travels back down through the layers. At each layer it's multiplied by more numbers. If those are smaller than 1, the signal gets smaller floor by floor, like a message whispered down a long line of people. The floors near the input get almost nothing and barely learn. That's the **vanishing gradient**.

A picture with the real numbers comes after the experiment below.

### ❓ What's a shortcut?

Don't replace the numbers with the layer's answer. **Add** the layer's answer to the numbers it was given:

```
without:   x → [layer] → out            out = layer(x)
with:      x → [layer] → + → out        out = x + layer(x)
           └────────────┘ the shortcut
```

Two good things happen. Going forward, the layer only has to learn a small correction to `x`, which is easier than rebuilding it. Going backward, the gradient gets a second road that skips the layer, so it doesn't have to pass through the layer's multiplication. That road is clear all the way down to the lowest floor.

Here's a tiny example with the first four numbers of the real row for `ἄ`, and one untrained layer.

👉 **Predict first:** will `x + layer(x)` be close to `x`, or far from it?

In [ ]:
import torch
from torch import nn

from homer_ai.tokenizer import CharTokenizer
from homer_ai.train_homer import BOOKS_DIR, load_books

tokenizer = CharTokenizer(load_books(BOOKS_DIR)[0])
torch.manual_seed(0)
embedding = nn.Embedding(tokenizer.vocab_size, 32)
layer = nn.Linear(32, 32)

letter = torch.tensor(tokenizer.encode("ἄ"))
row = embedding(letter)[0]  # the 32 numbers for ἄ
answer = layer(row)

print("x (first 4)          :", [round(v, 2) for v in row[:4].tolist()])
print("layer(x)             :", [round(v, 2) for v in answer[:4].tolist()])
print("x + layer(x)         :", [round(v, 2) for v in (row + answer)[:4].tolist()])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The layer's answer is a different set of numbers, but `x + layer(x)` still carries all of `x` inside it. So nothing from the input is lost, however badly the layer treats it.

</details>

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: build a deep stack, with a switch for shortcuts

In PyTorch a model is a class based on `nn.Module`. Its constructor (`__init__`) makes the layers, so PyTorch can find all their weights, and `forward` says what happens to the numbers that go through ([A_04](../A_toolbox/A_04_pytorch_in_this_repo.ipynb)). This is the book's experiment: five small layers, each followed by GELU. Four of them take 3 numbers in and give 3 out, and the last one goes from 3 numbers to 1. The switch `use_shortcut` decides whether a layer does `x + layer(x)` or just `layer(x)`. The last layer changes the shape (3 numbers to 1), so it can't have a shortcut.

👉 **Predict first:** with the same starting weights, will the two stacks give the same output?

In [ ]:
class DeepStack(nn.Module):
    def __init__(self, sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList(
            nn.Sequential(nn.Linear(sizes[i], sizes[i + 1]), nn.GELU())
            for i in range(len(sizes) - 1)
        )

    def forward(self, x):
        for layer in self.layers:
            out = layer(x)
            if self.use_shortcut and x.shape == out.shape:
                x = x + out  # the shortcut
            else:
                x = out
        return x


sizes = [3, 3, 3, 3, 3, 1]
sample = torch.tensor([[1.0, 0.0, -1.0]])  # three invented numbers for one letter

torch.manual_seed(123)
plain = DeepStack(sizes, use_shortcut=False)
torch.manual_seed(123)  # the same starting weights
shortcut = DeepStack(sizes, use_shortcut=True)
print("without:", plain(sample).item())
print("with:   ", shortcut(sample).item())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Different outputs from exactly the same weights, because the shortcuts changed what flows through. `nn.ModuleList` is a list that PyTorch knows holds layers, so it can find their weights. And `x.shape == out.shape` is the check that a shortcut is only added when the two have the same shape.

</details>

### ❓ Step 2: how big is the gradient in each layer?

We give each stack a target of 0, measure the mistake (mean squared error), run `backward()` ([01_03](../01_foundations/01_03_how_a_program_learns.ipynb)), and print the average size of the gradient of each layer's weights. Layer 1 is the one nearest the input.

👉 **Predict first:** with shortcuts, will layer 1's gradient be larger or smaller than without? By how much?

In [ ]:
def gradient_sizes(model):
    loss = nn.MSELoss()(model(sample), torch.tensor([[0.0]]))
    loss.backward()
    return [layer[0].weight.grad.abs().mean().item() for layer in model.layers]


without = gradient_sizes(plain)
with_shortcuts = gradient_sizes(shortcut)
print("layer   without shortcuts   with shortcuts")
for i, (a, b) in enumerate(zip(without, with_shortcuts, strict=True), start=1):
    print(f"  {i}        {a:.5f}            {b:.5f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Without shortcuts, the gradient in the layers near the input is tiny. Layer 1 gets about 0.0002, against about 0.005 in the last layer. That's 25 times less. It doesn't fall at every single step (layer 2 is a hair smaller than layer 1, because five small layers with random starting weights aren't perfectly even), but the trend is clear. With shortcuts, layer 1 gets about 0.22. That's roughly a thousand times bigger, and no layer is starved. The signal reaches the lowest floor. (Your numbers match ours because of `torch.manual_seed(123)`.)

</details>

### ❓ Can I see it?

Same numbers as bars, one pair per layer. The vertical axis is a **log** scale, because the two sets differ by a factor of about a thousand.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import BAD, GOOD, INK, MUTED, studio_axes, studio_figure

fig = studio_figure("Shortcuts", "The learning signal in each layer", (10, 6))
ax = studio_axes(fig)
positions = range(1, 6)
bar = {"width": 0.38}
left = [p - 0.2 for p in positions]
right = [p + 0.2 for p in positions]
label = {"ha": "center", "color": INK, "fontsize": 8, "family": "monospace"}
ax.bar(left, without, color=BAD, **bar, label="without shortcuts")
ax.bar(right, with_shortcuts, color=GOOD, **bar, label="with shortcuts")
for p, a, b in zip(positions, without, with_shortcuts, strict=True):
    ax.text(p - 0.2, a * 1.15, f"{a:.4f}", **label)
    ax.text(p + 0.2, b * 1.15, f"{b:.2f}", **label)
ax.set_yscale("log")
ax.set_ylim(top=3)
names = [f"layer {p}" for p in positions]
names[0] += " (input end)"
ax.set_xticks(list(positions), names, color=MUTED, fontsize=9)
ax.set_ylabel("average gradient size")
ax.legend(frameon=False, labelcolor=INK, loc="upper left")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The red bars (no shortcuts) shrink towards the input end, and the green ones stay high everywhere. That's why a tall stack of blocks needs shortcuts. Without them the lower floors would learn almost nothing.

</details>

**Look at:** how thick each arrow is. The mistake travels down from the output; the thicker the arrow, the more of the signal reaches that layer. On the right, the green loops are the shortcuts.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    BAD,
    GOOD,
    INK,
    MUTED,
    arrow,
    diagram_axes,
    stack_of_floors,
    studio_figure,
    tag,
)

fig = studio_figure("Shortcuts", "The learning signal, layer by layer", (11, 6.5))
ax = diagram_axes(fig, xlim=(0, 12))
top_size = max(*without, *with_shortcuts)
panels = [
    (3.0, "without shortcuts", BAD, without),
    (9.0, "with shortcuts", GOOD, with_shortcuts),
]
for x, name, color, sizes in panels:
    tag(ax, x, 6.0, name, color, 12, weight="bold")
    floors = [(f"layer {k}", "", MUTED) for k in range(1, 6)]
    ys = stack_of_floors(ax, x, 0.25, 2.0, floors, floor_h=0.5, gap=0.55)
    arrow(ax, (x, 5.7), (x, ys[4] + 0.3), "mistake", INK, 1.5, label_offset=(0.7, 0))
    for k in range(5):
        ratio = sizes[k] / top_size  # the width of the arrow
        local = sizes[k] / max(sizes)  # the colour: how this layer compares
        shade = ACCENT if local > 0.5 else (MUTED if local > 0.1 else BAD)
        if color == GOOD:
            shade = GOOD
        if k < 4:
            start, end = (x, ys[k + 1] - 0.3), (x, ys[k] + 0.3)
            arrow(ax, start, end, color=shade, width=1.5 + 7 * ratio**0.5, head=9)
        tag(ax, x + 1.2, ys[k], f"{sizes[k]:.4f}", shade, 10, ha="left", mono=True)
for k in (1, 2, 3, 4):  # a shortcut skips layers 1 to 4 (layer 5 is the last)
    y = 0.5 + (k - 1) * 1.05  # the middle of layer k
    arrow(ax, (8.3, y + 0.6), (8.3, y - 0.6), color=GOOD, width=2, curved=0.9, head=9)
tag(ax, 6.0, 3.0, "gradient size\nis written beside\neach layer", MUTED, 9)
plt.show()

### ❓ Does our `model.py` have shortcuts?

No, and it doesn't need them: its layers aren't stacked, so nothing fades. The block in [04_04](04_04_one_block.ipynb) uses two shortcuts. We don't change `model.py` here.

## 5. Recap

- In a deep stack the gradient fades on the way down, so the lowest layers barely learn (we measured about 0.0002 in layer 1).
- A shortcut adds a layer's input back to its output (`x + layer(x)`). With it, layer 1's gradient was about 0.22.
- Our `model.py` has no shortcuts. Every block has two.

**You can now:**

- add a shortcut around a layer: `x + layer(x)`
- measure the gradient that reaches the first layer, with and without shortcuts

**Next:** [04_04: One block](04_04_one_block.ipynb): putting the parts together.